# Laboratorio #8

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab8)
- [Data](https://www.ine.gob.gt/bases-de-datos/accidentes-de-transito/)

## Librerías

In [1]:
import pandas as pd
import pyreadstat
import sys
import os
import re
import glob
import web_scrapping
import subprocess
from contextlib import redirect_stdout
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, collect_set, sort_array

## Constantes

In [2]:
DATA_BASE = "/home/jovyan/work/data"
RAW_BASE = os.path.join(DATA_BASE, "raw")
MANIFEST_BASE = os.path.join(RAW_BASE, "manifest.log")

# Data que almacena los csv
HECHOS_BASE = os.path.join(DATA_BASE, "hechos")
VEHICULOS_BASE = os.path.join(DATA_BASE, "vehiculos")
FL_BASE = os.path.join(DATA_BASE, "fallecidos_lesionados")

# Caché
CACHE_BASE = "/home/jovyan/work/cache"
CACHE_WEB_SCRAPPING = "cache_web_scrapping.txt"
CACHE_TRANSFORM_SAV = "cache_transform_sav.txt"
CACHE_VARIABLES = "cache_variables.txt"
CACHE_SEPARATE = "cache_separate.txt"
CACHE_GENERATED_CSV = "cache_generated_csv.txt"

# Preparación archivos unificados
HECHOS_CSV = os.path.join(HECHOS_BASE, "hechos_combinado.csv")
VEHICULOS_CSV = os.path.join(VEHICULOS_BASE, "vehiculos_combinado.csv")
FL_CSV = os.path.join(FL_BASE, "fl_combinado.csv")

HECHOS_COLS = [
    "num_corre", "dia_ocu", "mes_ocu", "dia_sem_ocu", "hora_ocu",
    "g_hora", "depto_ocu", "mupio_ocu", "zona_ocu",
    "tipo_veh", "marca_veh", "color_veh", "modelo_veh", "año_ocu"
]

VEHICULOS_COLS = [
    "num_corre", "dia_ocu", "mes_ocu", "dia_sem_ocu", "hora_ocu",
    "g_hora", "depto_ocu", "mupio_ocu", "zona_ocu",
    "tipo_veh", "marca_veh", "color_veh", "modelo_veh", "año_ocu"
]

FL_COLS = [
    "num_corre", "dia_ocu", "mes_ocu", "dia_sem_ocu", "hora_ocu",
    "g_hora", "depto_ocu", "mupio_ocu", "zona_ocu",
    "tipo_veh", "marca_veh", "color_veh", "modelo_veh", "año_ocu"
]

# Aseguramos que existan las carpetas
os.makedirs(DATA_BASE, exist_ok=True)
os.makedirs(CACHE_BASE, exist_ok=True)

## Obtener data y procesarla

### Ver estructura de carpeta

In [3]:
def printTree(root, prefix=""):
    files = os.listdir(root)
    for i, f in enumerate(files):
        path = os.path.join(root, f)
        connector = "└── " if i == len(files) - 1 else "├── "
        print(prefix + connector + f)
        if os.path.isdir(path):
            extension = "    " if i == len(files) - 1 else "│   "
            printTree(path, prefix + extension)

### Función para ejecutar el script para "web_scrapping.py" obtener data

In [4]:
def runWebScrapping():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_WEB_SCRAPPING)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping.")
        return

    scriptPath = os.path.join(os.getcwd(), "web_scrapping.py")
    if not os.path.exists(scriptPath):
        print(f"No se encontró {scriptPath}")
        return

    print("Ejecutando web_scrapping.py ...")
    result = subprocess.run(["python", scriptPath], capture_output=True, text=True)

    # Mostrar salida en notebook
    print(result.stdout)
    if result.stderr:
        print("Errores:", result.stderr)

### Funciones para pasar archivos `.sav` a formato excel

In [5]:
def convertSavToXlsx(inputPath, outputPath):
    df, meta = pyreadstat.read_sav(inputPath)
    df.to_excel(outputPath, index=False)
    print(f"Convertido: {inputPath} -> {outputPath}")

In [6]:
def processSavFiles():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_TRANSFORM_SAV)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado la conversión de .sav a .xlsx.")
        return

    # Leer manifest.log de una vez
    manifest_path = MANIFEST_BASE
    manifest_lines = []
    if os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as mf:
            manifest_lines = mf.readlines()

    # Diccionario para actualizar manifest
    sav_to_xlsx = {}

    for root, dirs, files in os.walk(RAW_BASE):
        for f in files:
            if f.endswith(".sav"):
                inputFile = os.path.join(root, f)
                outputFile = inputFile.replace(".sav", ".xlsx")
                
                # Convertir
                convertSavToXlsx(inputFile, outputFile)

                # Borrar .sav
                os.remove(inputFile)
                print(f"\tEliminado: {inputFile}")

                # Guardar correspondencia
                sav_to_xlsx[f] = os.path.basename(outputFile)

    # Actualizar manifest.log
    if sav_to_xlsx and manifest_lines:
        updated_lines = []
        for line in manifest_lines:
            for sav_name, xlsx_name in sav_to_xlsx.items():
                if sav_name in line:
                    line = line.replace(sav_name, xlsx_name)
            updated_lines.append(line)

        with open(manifest_path, "w", encoding="utf-8") as mf:
            mf.writelines(updated_lines)
        print(f"Manifest actualizado: {manifest_path}")

    # Crear cache
    with open(cacheFilePath, "w") as cacheFile:
        cacheFile.write("Transformación de .sav completada.\n")
    print(f"Caché creado: {cacheFilePath}")

In [7]:
def runWebScrappingVariables():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_VARIABLES)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping de variables.")
        return

    # Ejecutar ws_variables.py desde el notebook
    scriptPath = os.path.join(os.getcwd(), "ws_variables.py")
    if not os.path.exists(scriptPath):
        print(f"No se encontró {scriptPath}")
        return

    print("Ejecutando ws_variables.py ...")
    result = subprocess.run(["python", scriptPath], capture_output=True, text=True)

    # Mostrar salida en notebook
    print(result.stdout)
    if result.stderr:
        print("Errores:", result.stderr)

### Separación de data raw y conversión a csv

In [8]:
def separateExcelToCsv():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_SEPARATE)
    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado la separación de Excel a CSV.")
        return

    # Aseguramos que existan las carpetas
    os.makedirs(HECHOS_BASE, exist_ok=True)
    os.makedirs(VEHICULOS_BASE, exist_ok=True)
    os.makedirs(FL_BASE, exist_ok=True)

    if not os.path.exists(MANIFEST_BASE):
        print(f"No se encontró {MANIFEST_BASE}")
        return

    # Leer manifest.log original
    with open(MANIFEST_BASE, "r", encoding="utf-8") as mf:
        lines = mf.readlines()

    # Diccionarios para almacenar los nuevos manifest
    new_manifest = {
        "hechos": [],
        "vehiculos": [],
        "fallecidos_lesionados": []
    }

    # Expresiones regulares para identificar tipo y año
    pattern = re.compile(r"(\w+): .*? -> .*/(\d{4})/([^/]+\.xlsx)")

    for line in lines:
        match = pattern.search(line)
        if not match:
            continue
        tipo, year, xlsx_file = match.groups()

        # Determinar carpeta de destino
        if tipo == "hechos":
            base_path = HECHOS_BASE
        elif tipo == "vehiculos":
            base_path = VEHICULOS_BASE
        elif tipo == "fallecidos_lesionados":
            base_path = FL_BASE
        else:
            continue

        # Crear nombre de CSV: tipo_año.csv o tipo_año_num.csv si ya existe
        csv_name = f"{tipo}_{year}.csv"
        csv_path = os.path.join(base_path, csv_name)
        i = 1
        while os.path.exists(csv_path):
            csv_name = f"{tipo}_{year}_{i}.csv"
            csv_path = os.path.join(base_path, csv_name)
            i += 1

        # Leer Excel y guardar a CSV
        xlsx_path = os.path.join(RAW_BASE, year, xlsx_file)
        if os.path.exists(xlsx_path):
            df = pd.read_excel(xlsx_path)
            df.to_csv(csv_path, index=False)
            print(f"Guardado CSV: {csv_path}")

            # Actualizar manifest del tipo
            new_manifest[tipo].append(f"{csv_name} -> {xlsx_file}\n")

    # Guardar manifest dentro de cada carpeta
    for tipo, entries in new_manifest.items():
        if entries:
            if tipo == "hechos":
                manifest_path = os.path.join(HECHOS_BASE, "manifest.log")
            elif tipo == "vehiculos":
                manifest_path = os.path.join(VEHICULOS_BASE, "manifest.log")
            else:
                manifest_path = os.path.join(FL_BASE, "manifest.log")
            
            with open(manifest_path, "w", encoding="utf-8") as mf:
                mf.writelines(entries)
            print(f"Manifest creado: {manifest_path}")

    # Crear cache
    with open(cacheFilePath, "w") as cacheFile:
        cacheFile.write("Separación de Excel a CSV completada.\n")
    print(f"Caché creado: {cacheFilePath}")

### Generación de dataset a utilizar (hechos, vehiculos, fl)

In [9]:
def extract_year_from_filename(filename):
    """Extrae el año del archivo, asumiendo formato *_YYYY.csv"""
    import re
    match = re.search(r'(\d{4})', filename)
    if match:
        return int(match.group(1))
    return None

In [10]:
def combineHechosCsv():
    """
    Combina todos los CSV de HECHOS_BASE en un solo CSV HECHOS_CSV,
    unificando nombres de columnas, agregando año_ocu y mostrando conteos.
    """
    all_files = [f for f in os.listdir(HECHOS_BASE) if f.endswith(".csv")]
    if not all_files:
        print(f"No se encontraron CSV en {HECHOS_BASE}")
        return

    combined_df = pd.DataFrame(columns=HECHOS_COLS)
    total_rows = 0

    for file in all_files:
        file_path = os.path.join(HECHOS_BASE, file)
        df = pd.read_csv(file_path)

        # Extraer año del nombre de archivo
        año = extract_year_from_filename(file)
        df["año_ocu"] = año

        # Normalizar nombres de columnas
        df.columns = [col.lower().replace("á","a").replace("é","e").replace("í","i")
                      .replace("ó","o").replace("ú","u").replace("ñ","n").strip() for col in df.columns]

        # Renombrar columnas
        rename_map = {
            "num_hecho": "num_corre", "num_corre": "num_corre", "núm_corre": "num_corre"
        }
        df.rename(columns=rename_map, inplace=True)

        # Seleccionar columnas comunes
        df = df[[c for c in HECHOS_COLS if c in df.columns]]

        print(f"{file}: {len(df)} filas")
        total_rows += len(df)
        combined_df = pd.concat([combined_df, df], ignore_index=True)

    print(f"Suma total de filas: {total_rows}")
    combined_df.to_csv(HECHOS_CSV, index=False)
    print(f"CSV combinado creado: {HECHOS_CSV}, tamaño final: {combined_df.shape}")

In [11]:
def combineVehiculosCsv():
    """
    Combina todos los CSV de VEHICULOS_BASE en un solo CSV VEHICULOS_CSV,
    unificando nombres de columnas, agregando año_ocu y mostrando conteos.
    """
    all_files = [f for f in os.listdir(VEHICULOS_BASE) if f.endswith(".csv")]
    if not all_files:
        print(f"No se encontraron CSV en {VEHICULOS_BASE}")
        return

    combined_df = pd.DataFrame(columns=VEHICULOS_COLS)
    total_rows = 0

    for file in all_files:
        file_path = os.path.join(VEHICULOS_BASE, file)
        df = pd.read_csv(file_path)

        # Extraer año del nombre de archivo
        año = extract_year_from_filename(file)
        df["año_ocu"] = año

        # Normalizar y renombrar columnas
        df.columns = [col.lower().replace("á","a").replace("é","e").replace("í","i")
                      .replace("ó","o").replace("ú","u").replace("ñ","n").strip() for col in df.columns]
        rename_map = {
            "num_hecho": "num_corre", "num_corre": "num_corre",
            "num_correlativo": "num_corre", "num_correlativo_base": "num_corre"
        }
        df.rename(columns=rename_map, inplace=True)

        df = df[[c for c in VEHICULOS_COLS if c in df.columns]]

        print(f"{file}: {len(df)} filas")
        total_rows += len(df)
        combined_df = pd.concat([combined_df, df], ignore_index=True)

    print(f"Suma total de filas: {total_rows}")
    combined_df.to_csv(VEHICULOS_CSV, index=False)
    print(f"CSV combinado creado: {VEHICULOS_CSV}, tamaño final: {combined_df.shape}")

In [12]:
def combineFlCsv():
    """
    Combina todos los CSV de FL_BASE en un solo CSV FL_CSV,
    unificando nombres de columnas, agregando año_ocu y mostrando conteos.
    """
    all_files = [f for f in os.listdir(FL_BASE) if f.endswith(".csv")]
    if not all_files:
        print(f"No se encontraron CSV en {FL_BASE}")
        return

    dfs = []
    total_rows = 0

    for file in all_files:
        file_path = os.path.join(FL_BASE, file)
        df = pd.read_csv(file_path)

        # Extraer año del nombre de archivo
        año = extract_year_from_filename(file)
        df["año_ocu"] = año

        # Normalizar columnas
        df.columns = [col.lower().replace("á","a").replace("é","e").replace("í","i")
                      .replace("ó","o").replace("ú","u").replace("ñ","n").strip() for col in df.columns]

        rename_map = {
            "num_hecho": "num_corre", "num_corre": "num_corre",
            "num_correlativo": "num_corre", "corre_base": "num_corre"
        }
        df.rename(columns=rename_map, inplace=True)

        df = df.loc[:, ~df.columns.duplicated()]
        df = df[[c for c in FL_COLS if c in df.columns]]

        print(f"{file}: {len(df)} filas")
        total_rows += len(df)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"Suma total de filas: {total_rows}")
    combined_df.to_csv(FL_CSV, index=False)
    print(f"CSV combinado creado: {FL_CSV}, tamaño final: {combined_df.shape}")

In [13]:
def runCombineCsv():
    """
    Ejecuta la combinación de CSV de HECHOS, VEHICULOS y FL si no existe el cache.
    Genera un manifest.log en data/ con todos los prints y un archivo de cache.
    """
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_GENERATED_CSV)
    manifestPath = os.path.join(DATA_BASE, "manifest.log")

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de combinación de CSV.")
        return

    # Abrir manifest.log para capturar todos los prints
    with open(manifestPath, "w", encoding="utf-8") as f:
        with redirect_stdout(f):
            print("=== Combinando HECHOS CSV ===")
            combineHechosCsv()
            print("\n=== Combinando VEHICULOS CSV ===")
            combineVehiculosCsv()
            print("\n=== Combinando FALLECIDOS/LESIONADOS CSV ===")
            combineFlCsv()
            print("\nProceso de combinación completado.")

    # Crear archivo de cache
    with open(cacheFilePath, "w") as f:
        f.write("Cache generado: combinación de CSV realizada\n")

    print(f"Proceso finalizado. Manifest guardado en {manifestPath}")
    print(f"Archivo de cache generado en {cacheFilePath}")

### Pipeline de preparación de data

In [14]:
runWebScrapping()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping.


In [15]:
processSavFiles()

Archivo de caché encontrado. Ya se ha ejecutado la conversión de .sav a .xlsx.


In [16]:
runWebScrappingVariables()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping de variables.


In [17]:
separateExcelToCsv()

Archivo de caché encontrado. Ya se ha ejecutado la separación de Excel a CSV.


In [19]:
runCombineCsv()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de combinación de CSV.


In [ ]:
# printTree(RAW_BASE)